# Exploration de la couche Silver

Cette exploration répond aux questions du persona — responsable de la mobilité urbaine : quels modes et créneaux concentrent le plus de comptages, et quels sites/axes présentent les volumes observés les plus importants ? Les volumes mesurés ne sont pas assimilés à une capacité routière ni à un taux de saturation.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F
from commun.spark_utils import LAKE, creer_session, format_tables

spark = creer_session("S03_ExplorationSilver", lac=True, delta=True, postgres=False)
spark.conf.set("spark.sql.session.timeZone", "Europe/Paris")
FMT = format_tables(spark)
SILVER_PATH = f"{LAKE}/silver/comptage_multimodal_paris"
silver = spark.read.format(FMT).load(SILVER_PATH)
print(f"Lignes Silver : {silver.count():,}")
silver.printSchema()

## Périmètre et qualité

Les premiers indicateurs aident à interpréter les graphiques. Les coordonnées manquantes excluent uniquement une ligne d'une éventuelle carte, pas des agrégations temporelles.

In [ ]:
silver.agg(
    F.count("*").alias("lignes"),
    F.countDistinct("identifiant_site").alias("sites"),
    F.countDistinct("mode_deplacement").alias("modes"),
    F.sum((~F.col("geo_connue")).cast("long")).alias("lignes_sans_coordonnees"),
    F.min("date_heure").alias("premier_comptage"),
    F.max("date_heure").alias("dernier_comptage")
).show(truncate=False)

totaux_modes = (silver.groupBy("mode_deplacement")
    .agg(F.sum("nombre_vehicules").alias("vehicules_comptes"),
         F.count("*").alias("observations"))
    .orderBy(F.desc("vehicules_comptes")))
totaux_modes.show(truncate=False)
modes_top = [r["mode_deplacement"] for r in totaux_modes.limit(5).collect()]

## Évolution mensuelle par mode

Le graphique agrège les comptages par mois et par mode. Il décrit les volumes relevés dans le jeu de données ; une variation peut aussi refléter les périodes de collecte ou la couverture des capteurs.

In [ ]:
mensuel = (silver.filter(F.col("mode_deplacement").isin(modes_top))
    .withColumn("mois_date", F.date_trunc("month", "date_heure"))
    .groupBy("mois_date", "mode_deplacement")
    .agg(F.sum("nombre_vehicules").alias("vehicules_comptes"))
    .toPandas())
if mensuel.empty:
    raise ValueError("Aucun agrégat mensuel disponible pour les modes sélectionnés.")
mensuel["mois_date"] = pd.to_datetime(mensuel["mois_date"])
fig, ax = plt.subplots(figsize=(12, 5))
for mode, serie in mensuel.groupby("mode_deplacement"):
    serie = serie.sort_values("mois_date")
    ax.plot(serie["mois_date"], serie["vehicules_comptes"], marker="o", markersize=3, label=mode)
ax.set(title="Comptages mensuels par mode", xlabel="Mois", ylabel="Véhicules comptés")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Mode", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

## Profil horaire des modes

La moyenne des comptages observés par heure facilite la comparaison des rythmes journaliers. L'indice Spark `jour_semaine` vaut 1 pour dimanche et 7 pour samedi.

In [ ]:
horaire = (silver.filter(F.col("mode_deplacement").isin(modes_top))
    .groupBy("heure", "mode_deplacement")
    .agg(F.avg("nombre_vehicules").alias("moyenne_vehicules"))
    .orderBy("heure")
    .toPandas())
fig, ax = plt.subplots(figsize=(10, 5))
for mode, serie in horaire.groupby("mode_deplacement"):
    serie = serie.sort_values("heure")
    ax.plot(serie["heure"], serie["moyenne_vehicules"], marker="o", label=mode)
ax.set(title="Moyenne des comptages selon l'heure", xlabel="Heure de la journée", ylabel="Véhicules par observation")
ax.set_xticks(range(24))
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Mode", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

## Sites et types de voie aux volumes observés les plus élevés

Ce classement sert à repérer les sites à examiner. Il additionne les relevés de toute la période disponible : il ne corrige ni la durée d'observation ni la capacité des axes et ne permet donc pas, seul, de conclure à une saturation.

In [ ]:
sites_top = (silver
    .withColumn("site_affiche", F.coalesce("nom_du_site", F.concat(F.lit("Site "), F.col("identifiant_site"))))
    .groupBy("identifiant_site", "site_affiche", "type_voie")
    .agg(F.sum("nombre_vehicules").alias("vehicules_comptes"))
    .orderBy(F.desc("vehicules_comptes"))
    .limit(15)
    .toPandas())
sites_top["libelle"] = sites_top["site_affiche"] + " — " + sites_top["type_voie"].fillna("voie inconnue")
sites_top = sites_top.sort_values("vehicules_comptes")
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(sites_top["libelle"], sites_top["vehicules_comptes"], color="#3973ac")
ax.set(title="15 sites aux volumes cumulés les plus élevés", xlabel="Véhicules comptés (somme)", ylabel="Site et type de voie")
ax.set_xlim(left=0)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## Intensité moyenne selon le jour et l'heure

La carte de chaleur résume la moyenne par observation, pas le total annuel. Les couleurs servent à repérer les créneaux à approfondir ; vérifier la couverture des capteurs avant toute interprétation causale.

In [ ]:
jours = {1: "Dim", 2: "Lun", 3: "Mar", 4: "Mer", 5: "Jeu", 6: "Ven", 7: "Sam"}
profil = (silver.groupBy("jour_semaine", "heure")
    .agg(F.avg("nombre_vehicules").alias("moyenne"))
    .toPandas())
matrice = profil.pivot(index="jour_semaine", columns="heure", values="moyenne").reindex(index=range(1, 8), columns=range(24))
fig, ax = plt.subplots(figsize=(14, 4))
image = ax.imshow(matrice, aspect="auto", origin="upper", cmap="YlOrRd")
ax.set_yticks(range(7), [jours[i] for i in range(1, 8)])
ax.set_xticks(range(24))
ax.set(title="Moyenne des véhicules comptés par jour et heure", xlabel="Heure", ylabel="Jour")
fig.colorbar(image, ax=ax, label="Véhicules par observation")
fig.tight_layout()
plt.show()